# For Reference

### AT - Ambient Temperature
### V - Exhaust Vacuum
### AP - Ambient pressure
### RH - Relative humidity
### PE - Electric  power Output

## Importing Libraries and Loading Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv(r'../data/dataset.csv')

## Checking the structure of the dataset

In [ ]:
# Basic information
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types and non-null counts:")
df.info()

print("\nFirst 5 rows:")
display(df.head())

## Inspecting the data for missing values and Duplicates

In [ ]:
# Check missing values
print("Missing values in each column:")
print(df.isnull().sum())

print("\nTotal missing values:", df.isnull().sum().sum())

# Check duplicate rows
print("\nNumber of duplicate rows:", df.duplicated().sum())

## Statistical Analysis of the Dataset

In [ ]:
# Mean
print("Mean:")
print(df.mean())

# Median
print("Median:")
print(df.median())

# Standard deviation
print("Standard Deviation:")
print(df.std())

# Covariance
print("Covariance:")
print(df.cov())

# Correlation
print("Correlation:")
print(df.corr())

# Correlation heatmap
plt.figure(figsize=(8,6))
sns.heatmap(df.corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

## Creating Project Directories

In [ ]:
import os
folders = [
    "../plots/EDA",
    "../plots/Feature_Selection",
    "../plots/Model_Evaluation"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Folders created successfully!")

## Data Preparation and Train-Test Split

In [ ]:
# Duplicate values
print("Duplicates:", df.duplicated().sum())

# Remove duplicates if any
#df = df.drop_duplicates()
print(f"Dataset size after removing duplicates: {df.shape}")

X = df[['AT', 'V', 'AP', 'RH']]
y = df['PE']

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

print("Total samples:", len(X))
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("Training percentage:",
      len(X_train) / len(X) * 100)

print("Testing percentage:",
      len(X_test) / len(X) * 100)

## Exploratory Data Analysis

In [ ]:
# Detect outliers using IQR

for col in df.columns:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[
        (df[col] < lower) | (df[col] > upper)
    ]

    print(col, ":", len(outliers), "outliers")

    # Histograms of all features

df.hist(
    figsize=(12, 10),
    bins=30,
    edgecolor="black"
)

plt.suptitle("Distribution of Power Plant Variables")
plt.tight_layout()

plt.savefig("../plots/EDA/histograms.png")
plt.show()

# Individual boxplots

for col in df.columns:

    plt.figure(figsize=(7, 4))

    sns.boxplot(x=df[col])

    plt.title("Boxplot of " + col)
    plt.xlabel(col)

    plt.savefig("../plots/EDA/boxplot_" + col + ".png")
    plt.show()


features = ["AT", "V", "AP", "RH"]

for col in features:

    plt.figure(figsize=(7, 5))

    plt.scatter(df[col], df["PE"], alpha=0.4)

    plt.xlabel(col)
    plt.ylabel("Electrical Power Output (PE)")
    plt.title(col + " vs Electrical Power Output")

    plt.savefig("../plots/EDA/scatter_" + col + ".png")
    plt.show()



print("EDA plots saved successfully!")
print(os.listdir("../plots/EDA"))

## Feature Combination Comparison

In [ ]:
from itertools import combinations
features = ['AT', 'V', 'AP', 'RH']

feature_combinations = []

for r in range(1, 5):
    feature_combinations.extend(combinations(features, r))

print("Total combinations:", len(feature_combinations))

for i, combo in enumerate(feature_combinations, 1):
    print(i, ":", combo)

results = []

for combo in feature_combinations:
    X_train_selected = X_train[list(combo)]
    X_test_selected = X_test[list(combo)]

    model = LinearRegression()
    model.fit(X_train_selected, y_train)

    y_pred = model.predict(X_test_selected)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    results.append({
        'Features': ' + '.join(combo),
        'Number of Features': len(combo),
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'R2 Score': r2
    })

results_df = pd.DataFrame(results)

print(results_df)

results_df = results_df.sort_values(by='RMSE')

print("Models ranked by RMSE:")
display(results_df)

best_model_features = results_df.iloc[0]['Features']

print("Best feature combination:", best_model_features)

results_df.to_csv(
    "../plots/Feature_Selection/feature_comparison_results.csv",
    index=False
)



plt.figure(figsize=(14, 6))

plt.bar(results_df['Features'], results_df['RMSE'])

plt.xlabel("Feature Combinations")
plt.ylabel("RMSE")
plt.title("RMSE Comparison of 15 Linear Regression Models")

plt.xticks(rotation=60, ha='right')
plt.tight_layout()

plt.savefig(
    "../plots/Feature_Selection/RMSE_comparison.png"
)

plt.show()


plt.figure(figsize=(14, 6))

plt.bar(results_df['Features'], results_df['R2 Score'])

plt.xlabel("Feature Combinations")
plt.ylabel("R² Score")
plt.title("R² Score Comparison of 15 Linear Regression Models")

plt.xticks(rotation=60, ha='right')
plt.tight_layout()

plt.savefig(
    "../plots/Feature_Selection/R2_comparison.png"
)

plt.show()

## Final Model Evaluation

In [ ]:
# Select the best feature combination
best_row = results_df.sort_values(by="RMSE").iloc[0]

best_features = best_row["Features"].split(" + ")

print("Best features:", best_features)

# Prepare training and testing data
X_train_best = X_train[best_features]
X_test_best = X_test[best_features]

# Train the final model
final_model = LinearRegression()
final_model.fit(X_train_best, y_train)

# Make predictions
y_pred_best = final_model.predict(X_test_best)


mae = mean_absolute_error(y_test, y_pred_best)
mse = mean_squared_error(y_test, y_pred_best)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred_best)

print("Final Model Performance")
print("-----------------------")
print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R2 Score:", r2)


from sklearn.model_selection import cross_validate

cv_results = cross_validate(
    LinearRegression(),
    X_train_best,
    y_train,
    cv=5,
    scoring={
        "MAE": "neg_mean_absolute_error",
        "RMSE": "neg_root_mean_squared_error",
        "R2": "r2"
    }
)

print("5-Fold Cross Validation Results")
print("--------------------------------")

print("Average MAE:", -cv_results["test_MAE"].mean())
print("Average RMSE:", -cv_results["test_RMSE"].mean())
print("Average R2:", cv_results["test_R2"].mean())



plt.figure(figsize=(7, 5))

plt.scatter(y_test, y_pred_best, alpha=0.5)

# Ideal prediction line
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    linestyle="--"
)

plt.xlabel("Actual Power Output (MW)")
plt.ylabel("Predicted Power Output (MW)")
plt.title("Actual vs Predicted Power Output")

plt.tight_layout()

plt.savefig(
    "../plots/Model_Evaluation/actual_vs_predicted.png"
)

plt.show()



residuals = y_test - y_pred_best

plt.figure(figsize=(7, 5))

plt.scatter(y_pred_best, residuals, alpha=0.5)

plt.axhline(y=0, linestyle="--")

plt.xlabel("Predicted Power Output (MW)")
plt.ylabel("Residuals (MW)")
plt.title("Residuals vs Predicted Values")

plt.tight_layout()

plt.savefig(
    "../plots/Model_Evaluation/residuals_vs_predicted.png"
)

plt.show()



plt.figure(figsize=(7, 5))

plt.hist(residuals, bins=30, edgecolor="black")

plt.xlabel("Residuals (MW)")
plt.ylabel("Frequency")
plt.title("Distribution of Prediction Errors")

plt.tight_layout()

plt.savefig(
    "../plots/Model_Evaluation/residual_histogram.png"
)

plt.show()



coefficient_df = pd.DataFrame({
    "Feature": best_features,
    "Coefficient": final_model.coef_
})

print(coefficient_df)
print("Intercept:", final_model.intercept_)




plt.figure(figsize=(8, 5))

plt.bar(
    coefficient_df["Feature"],
    coefficient_df["Coefficient"]
)

plt.xlabel("Features")
plt.ylabel("Regression Coefficient")
plt.title("Regression Coefficients of Best Model")

plt.tight_layout()

plt.savefig(
    "../plots/Model_Evaluation/regression_coefficients.png"
)

plt.show()



prediction_df = pd.DataFrame({
    "Actual PE": y_test,
    "Predicted PE": y_pred_best,
    "Residual": residuals
})

prediction_df.to_csv(
    "../plots/Model_Evaluation/predictions.csv",
    index=False
)

display(prediction_df.head(10))

## Final Results and Model Evaluation

In [ ]:
print("Dataset Shape:", df.shape)
print("\nFeatures:")
print(df.columns.tolist())

print("\nDescriptive Statistics:")
display(df.describe())


print("Missing Values:", df.isnull().sum().sum())
print("Duplicate Rows:", df.duplicated().sum())

print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))

display(
    results_df.sort_values(by="RMSE").reset_index(drop=True)
)


best_row = results_df.loc[results_df["RMSE"].idxmin()]

print("Best Feature Combination:", best_row["Features"])
print("MAE:", best_row["MAE"])
print("MSE:", best_row["MSE"])
print("RMSE:", best_row["RMSE"])
print("R2 Score:", best_row["R2 Score"])



print("Final Model Evaluation")
print("----------------------")
print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R2 Score:", r2)


print("5-Fold Cross Validation")
print("-----------------------")

print("Average MAE:", -cv_results["test_MAE"].mean())
print("Average RMSE:", -cv_results["test_RMSE"].mean())
print("Average R2:", cv_results["test_R2"].mean())